# 04. Test Set Analysis: Understanding the Competition Task

## Question: Are these random (user, song) pairs OR Deezer's specific recommendations?

This matters because it changes what features predict the outcome.

**If random pairs:** Use popularity + generic features  
**If Deezer's recommendations:** Use recommendation context + user affinity

## 1. Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from google.colab import drive
    #drive.mount('/content/gdrive')
    #base_path = '/content/gdrive/MyDrive/Recommender Systems (Project)/'
    drive.mount('/content/drive', force_remount=True)
    base_path = '/content/drive/MyDrive/Music Recommender System/Recommender Systems (Project)/'
except:
    base_path = './'

df_train = pd.read_csv(base_path + 'train_processed.csv')
df_test = pd.read_csv(base_path + 'test_processed.csv')

print(f'Train: {df_train.shape}')
print(f'Test: {df_test.shape}')

Mounted at /content/drive
Train: (7558184, 15)
Test: (19918, 21)


In [ ]:
# add features (for baseline models features were created after spliting to avoid leakage)


# 2. Convert timestamps
df_train['listen_datetime'] = pd.to_datetime(df_train['ts_listen'], unit='s')
df_train['listen_hour'] = df_train['listen_datetime'].dt.hour
df_train['listen_dayofweek'] = df_train['listen_datetime'].dt.dayofweek
df_train['listen_month'] = df_train['listen_datetime'].dt.month

# 3. Calculate song age
df_train['release_datetime'] = pd.to_datetime(df_train['release_date'].astype(str), format='%Y%m%d')
df_train['song_age_days'] = (df_train['listen_datetime'] - df_train['release_datetime']).dt.days

# Replace negative song_age
df_train.loc[df_train['song_age_days'] < 0, 'song_age_days'] = 755.0

# 4. Replace duration = 0
df_train.loc[df_train['media_duration'] == 0, 'media_duration'] = 222.0

# 5. Add popularity features
user_listen_count = df_train.groupby('user_id').size().reset_index(name='user_listen_count')
item_listen_count = df_train.groupby("media_id").size().reset_index(name="item_listen_count")
artist_listen_count = df_train.groupby("artist_id").size().reset_index(name="artist_listen_count")
genre_listen_count = df_train.groupby("genre_id").size().reset_index(name="genre_listen_count")

df_train = df_train.merge(user_listen_count, on='user_id', how='left')
df_train = df_train.merge(item_listen_count, on='media_id', how='left')
df_train = df_train.merge(artist_listen_count, on='artist_id', how='left')
df_train = df_train.merge(genre_listen_count, on='genre_id', how='left')

# 6. Drop original timestamp columns
df_train = df_train.drop(columns=['ts_listen', 'listen_datetime', 'release_date', 'release_datetime'])

print(f"Test processed shape: {df_train.shape}")
print(f"Test columns: {df_train.columns.tolist()}")

Test processed shape: (7558184, 21)
Test columns: ['genre_id', 'media_id', 'album_id', 'context_type', 'platform_name', 'platform_family', 'media_duration', 'listen_type', 'user_gender', 'user_id', 'artist_id', 'user_age', 'is_listened', 'listen_hour', 'listen_dayofweek', 'listen_month', 'song_age_days', 'user_listen_count', 'item_listen_count', 'artist_listen_count', 'genre_listen_count']


## 2. Test Set Structure Analysis

In [ ]:
print('='*70)
print('TEST SET STRUCTURE')
print('='*70)

n_test_rows = len(df_test)
n_unique_users = df_test['user_id'].nunique()
n_unique_items = df_test['media_id'].nunique()
avg_items_per_user = n_test_rows / n_unique_users

print(f'\nTest set size: {n_test_rows:,} rows')
print(f'Unique users: {n_unique_users:,}')
print(f'Unique items (songs): {n_unique_items:,}')
print(f'Average items per user: {avg_items_per_user:.2f}')

# Distribution of items per user
items_per_user = df_test.groupby('user_id').size()
print(f'\nDistribution of items per user:')
print(f'  Min: {items_per_user.min()}')
print(f'  Max: {items_per_user.max()}')
print(f'  Mean: {items_per_user.mean():.2f}')
print(f'  Median: {items_per_user.median():.0f}')
print(f'  Std: {items_per_user.std():.2f}')

TEST SET STRUCTURE

Test set size: 19,918 rows
Unique users: 19,918
Unique items (songs): 9,732
Average items per user: 1.00

Distribution of items per user:
  Min: 1
  Max: 1
  Mean: 1.00
  Median: 1
  Std: 0.00


## 3. Key Question: Does Every User Have Exactly 1 Item?

In [ ]:
print('\n' + '='*70)
print('CRITICAL CHECK: Items per user distribution')
print('='*70)

items_distribution = items_per_user.value_counts().sort_index()
print(f'\nValue Counts:')
print(items_distribution)

if (items_per_user == 1).all():
    print('\n✓ FINDING: Every user has EXACTLY 1 item in test set')
    print('   This suggests: Deezer recommended ONE song per user')
    print('   Your task: Predict if they will listen to that ONE song')
elif (items_per_user == 2).all():
    print('\n✓ FINDING: Every user has EXACTLY 2 items in test set')
    print('   This suggests: Maybe 2 recommendations or A/B test')
else:
    print('\n⚠️ FINDING: Unequal distribution of items per user')
    print('   This suggests: Random (user, song) pairs')


CRITICAL CHECK: Items per user distribution

Value Counts:
1    19918
Name: count, dtype: int64

✓ FINDING: Every user has EXACTLY 1 item in test set
   This suggests: Deezer recommended ONE song per user
   Your task: Predict if they will listen to that ONE song


## 4. Item Uniqueness Analysis

In [ ]:
print('\n' + '='*70)
print('ITEM UNIQUENESS ANALYSIS')
print('='*70)

# Check if items overlap with training data
train_items = set(df_train['media_id'].unique())
test_items = set(df_test['media_id'].unique())

items_in_train = len(test_items & train_items)
items_not_in_train = len(test_items - train_items)

print(f'\nTest items in training data: {items_in_train:,} ({100*items_in_train/n_unique_items:.1f}%)')
print(f'Test items NOT in training: {items_not_in_train:,} ({100*items_not_in_train/n_unique_items:.1f}%)')

# How many times does each item appear in test?
item_frequency = df_test.groupby('media_id').size()
print(f'\nItem frequency in test set:')
print(f'  Min appearances: {item_frequency.min()}')
print(f'  Max appearances: {item_frequency.max()}')
print(f'  Mean appearances: {item_frequency.mean():.2f}')

if (item_frequency == 1).all():
    print('\n✓ FINDING: Every item appears EXACTLY ONCE in test set')
    print('   This suggests: Each user got a unique recommendation')
else:
    print('\n⚠️ FINDING: Items appear multiple times')
    print('   This suggests: Deezer recommends same song to multiple users')


ITEM UNIQUENESS ANALYSIS

Test items in training data: 9,397 (96.6%)
Test items NOT in training: 335 (3.4%)

Item frequency in test set:
  Min appearances: 1
  Max appearances: 54
  Mean appearances: 2.05

⚠️ FINDING: Items appear multiple times
   This suggests: Deezer recommends same song to multiple users


## 5. User Popularity: Are Popular Users Overrepresented?

In [ ]:
print('\n' + '='*70)
print('USER ACTIVITY ANALYSIS')
print('='*70)

# In test: are these active or inactive users?
test_user_activity = df_test['user_listen_count'].describe()
train_user_activity = df_train['user_listen_count'].describe()

print('\nUser listening history in training data:')
print(train_user_activity)

print('\nTest users listening history (from training):')
print(test_user_activity)

# Compare
print(f'\nComparison:')
print(f'  Train avg: {train_user_activity["mean"]:.0f} listens')
print(f'  Test avg:  {test_user_activity["mean"]:.0f} listens')

if test_user_activity['mean'] > train_user_activity['mean']:
    print('\n✓ Test users are MORE active than average')
    print('  Suggests: Deezer selected active users for evaluation')
else:
    print('\n⚠️ Test users are LESS active than average')
    print('  Suggests: Random user selection')


USER ACTIVITY ANALYSIS

User listening history in training data:
count    7.558184e+06
mean     1.060209e+03
std      9.469270e+02
min      1.000000e+00
25%      4.030000e+02
50%      7.880000e+02
75%      1.402000e+03
max      6.317000e+03
Name: user_listen_count, dtype: float64

Test users listening history (from training):
count    18153.000000
mean       333.088030
std        417.004629
min          1.000000
25%         65.000000
50%        190.000000
75%        443.000000
max       5237.000000
Name: user_listen_count, dtype: float64

Comparison:
  Train avg: 1060 listens
  Test avg:  333 listens

⚠️ Test users are LESS active than average
  Suggests: Random user selection


## 6. Item Popularity: Are Popular Songs Recommended?

In [ ]:
print('\n' + '='*70)
print('ITEM (SONG) POPULARITY ANALYSIS')
print('='*70)

test_item_popularity = df_test['item_listen_count'].describe()
train_item_popularity = df_train['item_listen_count'].describe()

print('\nAll items popularity in training data:')
print(train_item_popularity)

print('\nTest set items popularity (from training):')
print(test_item_popularity)

# Compare
print(f'\nComparison:')
print(f'  Train avg: {train_item_popularity["mean"]:.0f} listens')
print(f'  Test avg:  {test_item_popularity["mean"]:.0f} listens')

if test_item_popularity['mean'] > train_item_popularity['mean']:
    print('\n✓ Test songs are MORE POPULAR than average')
    print('  Suggests: Deezer recommends popular songs (good strategy!)')
else:
    print('\n⚠️ Test songs are LESS popular than average')
    print('  Suggests: Could be niche recommendations or random')


ITEM (SONG) POPULARITY ANALYSIS

All items popularity in training data:
count    7.558184e+06
mean     1.564361e+03
std      2.817865e+03
min      1.000000e+00
25%      5.000000e+01
50%      3.860000e+02
75%      1.784000e+03
max      2.014700e+04
Name: item_listen_count, dtype: float64

Test set items popularity (from training):
count    19404.000000
mean      1286.948258
std       2002.667019
min          1.000000
25%         86.000000
50%        514.000000
75%       1688.000000
max      17079.000000
Name: item_listen_count, dtype: float64

Comparison:
  Train avg: 1564 listens
  Test avg:  1287 listens

⚠️ Test songs are LESS popular than average
  Suggests: Could be niche recommendations or random


## 7. Context Analysis: Do Recommendations Match User Preferences?

In [ ]:
print('\n' + '='*70)
print('RECOMMENDATION CONTEXT ANALYSIS')
print('='*70)

# What genres does each user listen to?
print('\nGenre distribution in training data:')
print(df_train['genre_id'].value_counts().head(10))

# What genres are recommended?
print('\nGenre distribution in test set:')
print(df_test['genre_id'].value_counts().head(10))

# Platform analysis
print('\nPlatform distribution in training data:')
print(df_train['platform_name'].value_counts())

print('\nPlatform distribution in test set:')
print(df_test['platform_name'].value_counts())

# Time analysis
print('\nListen hour distribution in training (peak hours):')
print(df_train['listen_hour'].value_counts().head(5))

print('\nTest listen hour distribution (peak hours):')
print(df_test['listen_hour'].value_counts().head(5))


RECOMMENDATION CONTEXT ANALYSIS

Genre distribution in training data:
genre_id
0       3666577
7        929394
10       288223
25       268998
27       187934
14       177095
734      141060
297      137967
77       101092
2744      84687
Name: count, dtype: int64

Genre distribution in test set:
genre_id
0       8741
7       2592
25       794
10       743
27       703
14       578
734      528
297      471
1130     293
77       278
Name: count, dtype: int64

Platform distribution in training data:
platform_name
0    5083321
1    1373597
2    1101266
Name: count, dtype: int64

Platform distribution in test set:
platform_name
0    12516
1     5952
2     1450
Name: count, dtype: int64

Listen hour distribution in training (peak hours):
listen_hour
17    560859
16    545387
18    485886
15    481803
14    452195
Name: count, dtype: int64

Test listen hour distribution (peak hours):
listen_hour
17    1733
18    1573
16    1519
15    1262
19    1225
Name: count, dtype: int64


## 8. Final Diagnosis

In [ ]:
print('\n' + '='*70)
print('FINAL DIAGNOSIS')
print('='*70)

print('\nEvidence Summary:')
print(f'1. Items per user: {avg_items_per_user:.2f} (1.0 = each user has 1 rec)')
print(f'2. Each item appears: {item_frequency.mean():.2f}x (1.0 = unique recommendations)')
print(f'3. Test users activity: {test_user_activity["mean"]:.0f} vs train {train_user_activity["mean"]:.0f}')
print(f'4. Test items popularity: {test_item_popularity["mean"]:.0f} vs train {train_item_popularity["mean"]:.0f}')

print('\n' + '='*70)

if (items_per_user == 1).all() and (item_frequency == 1).all():
    print('\n🎯 CONCLUSION: DEEZER RECOMMENDATIONS TASK')
    print('\nEach user has EXACTLY ONE recommended song.')
    print('Your task: Predict if they will listen to it.')
    print('\nIMPLICATIONS:')
    print('  - NOT random (user, song) pairs')
    print('  - These are DEEZER\'S ACTUAL RECOMMENDATIONS')
    print('  - Your features must capture recommendation context')
    print('  - User affinity to this SPECIFIC recommendation matters')
    print('\nFEATURES THAT MATTER:')
    print('  ✓ How well does this user like THIS genre?')
    print('  ✓ How well does this user like THIS artist?')
    print('  ✓ Is this song fresh or old? Does this user like old/new?')
    print('  ✓ Recommendation context (platform, time, etc.)')
    print('  ✓ User\'s past behavior on SIMILAR songs')

else:
    print('\n🔀 CONCLUSION: RANDOM PAIRS TASK')
    print('\nThese appear to be random (user, song) pairs.')
    print('Your task: Predict listening probability for any pair.')
    print('\nIMPLICATIONS:')
    print('  - Generic popularity signals matter')
    print('  - User/item co-occurrence matters')
    print('  - Collaborative filtering is key approach')


FINAL DIAGNOSIS

Evidence Summary:
1. Items per user: 1.00 (1.0 = each user has 1 rec)
2. Each item appears: 2.05x (1.0 = unique recommendations)
3. Test users activity: 333 vs train 1060
4. Test items popularity: 1287 vs train 1564


🔀 CONCLUSION: RANDOM PAIRS TASK

These appear to be random (user, song) pairs.
Your task: Predict listening probability for any pair.

IMPLICATIONS:
  - Generic popularity signals matter
  - User/item co-occurrence matters
  - Collaborative filtering is key approach


## 9. Next Steps Based on Diagnosis

In [ ]:
print('\nBased on your diagnosis, next steps should be:')
print('\nIf DEEZER RECOMMENDATIONS:')
print('  1. Build user-genre affinity (does user like this genre?)')
print('  2. Build user-artist affinity (does user like this artist?)')
print('  3. Analyze user behavior on similar songs')
print('  4. Consider recommendation recency (fresh vs old)')
print('  5. Use context features (platform, time matching user patterns)')
print('\nIf RANDOM PAIRS:')
print('  1. Use collaborative filtering (user-item similarity)')
print('  2. Build user-user and item-item embeddings')
print('  3. Focus on popularity signals')
print('  4. Matrix factorization approaches')


Based on your diagnosis, next steps should be:

If DEEZER RECOMMENDATIONS:
  1. Build user-genre affinity (does user like this genre?)
  2. Build user-artist affinity (does user like this artist?)
  3. Analyze user behavior on similar songs
  4. Consider recommendation recency (fresh vs old)
  5. Use context features (platform, time matching user patterns)

If RANDOM PAIRS:
  1. Use collaborative filtering (user-item similarity)
  2. Build user-user and item-item embeddings
  3. Focus on popularity signals
  4. Matrix factorization approaches
